In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..")))

In [2]:
from Composant3.data.load import load_training_pairs
from Composant3.backtest.walk_forward import run

df = load_training_pairs()
results = run(df)
print(results.to_string(index=False))


--- Pli test 2022 | train [2021] ---


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:574: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


  Best alpha: 100 | CV MAE: 0.0141


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:574: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:574: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(
C:\Users\user\AppData\Roaming\Python\Python313\site-packages\sklearn\impute\_base.py:574: FutureWarning: Currently, when `keep_empty_feature=False` and `strategy="constant"`, empty features are not dropped. This behaviour will change in version 1.8. Set `keep_empty_feature=True` to preserve this behaviour.
  warnings.warn(


  Best params: {'model__alpha': 0.01, 'model__l1_ratio': 0.3} | CV MAE: 0.0147
  Quantile 10% trained
  Quantile 90% trained
  Quantile coverage: {'coverage': 0.8372, 'avg_width': 0.0675}

--- Pli test 2023 | train [2021, 2022] ---
  Best alpha: 200 | CV MAE: 0.0144
  Best params: {'model__alpha': 0.001, 'model__l1_ratio': 0.1} | CV MAE: 0.0147
  Quantile 10% trained
  Quantile 90% trained
  Quantile coverage: {'coverage': 0.7961, 'avg_width': 0.0621}

--- Pli test 2024 | train [2021, 2022, 2023] ---
  Best alpha: 200 | CV MAE: 0.0138
  Best params: {'model__alpha': 0.001, 'model__l1_ratio': 0.1} | CV MAE: 0.0142
  Quantile 10% trained
  Quantile 90% trained
  Quantile coverage: {'coverage': 0.8039, 'avg_width': 0.0636}
         model  test_season   n    mae   rmse
         ridge         2022 393 0.0143 0.0183
   elastic_net         2022 393 0.0148 0.0187
mean_reversion         2022 393 0.0154 0.0203
     last_year         2022 393 0.0156 0.0215
     age_curve         2022 393 0.3994 0

In [3]:
from Composant3.evaluation.metrics import by_age_bucket, by_tier
from Composant3.baselines.mean_reversion import predict as mr_predict

# On prend le pli 3 (le plus récent) comme exemple
train = df[df["season_year"].isin([2021, 2022, 2023])]
test  = df[df["season_year"] == 2024]

preds = mr_predict(train, test)
y_true = test["target_pie"].values

print("=== Par tranche d'âge ===")
print(by_age_bucket(y_true, preds, test["age_at_date"]).to_string(index=False))

print("\n=== Par valuation tier ===")
print(by_tier(y_true, preds, test["valuation_tier"]).to_string(index=False))

=== Par tranche d'âge ===
age_bucket   n    mae   rmse
       ≤22  63 0.0160 0.0220
     23-25 131 0.0150 0.0195
     26-28 100 0.0165 0.0223
     29-32  74 0.0132 0.0173
       33+  40 0.0172 0.0218

=== Par valuation tier ===
   valuation_tier   n    mae   rmse
       low_sample  87 0.0148 0.0187
         reliable 285 0.0143 0.0185
replacement_level  36 0.0263 0.0343


In [4]:
df.isnull().sum()[df.isnull().sum() > 0]

weight         1
bmi            1
trend_pts    568
trend_pie    568
dtype: int64

In [ ]:
from sqlalchemy import create_engine, text, inspect
from Composant3.config import DB_URL
import pandas as pd

engine = create_engine(DB_URL)

# 1. Toutes les tables
with engine.connect() as conn:
    tables = pd.read_sql(text("""
        SELECT table_name 
        FROM information_schema.tables 
        WHERE table_schema = 'public'
        ORDER BY table_name
    """), conn)
print(tables)

# 2. Colonnes de training_pairs
with engine.connect() as conn:
    cols = pd.read_sql(text("""
        SELECT column_name, data_type 
        FROM information_schema.columns 
        WHERE table_name = 'training_pairs'
        ORDER BY ordinal_position
    """), conn)
print(cols)

# 3. Aperçu des données
with engine.connect() as conn:
    sample = pd.read_sql(text("SELECT * FROM training_pairs LIMIT 5"), conn)
print(sample)

NameError: name 'engine' is not defined